# UMLS-first MIMIC-IV discharge property graph model grid

This notebook builds LlamaIndex property graphs from a small MIMIC-IV discharge-note subset and compares model combinations.

For each run, the same generation model is used to build the property graph and answer the evaluation questions. Each generation model is paired with each embedding model, and the notebook collects an interactive HTML graph, a static JPEG plot, per-combination results, and a manifest.

The comparison grid is controlled by Python lists in the setup cell, so the `.env` file does not need sweep variables.


In [ ]:
from __future__ import annotations

import asyncio
import os
import sys
from pathlib import Path

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "main.py").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "main.py").exists():
            repo_root = parent
            break

sys.path.insert(0, str(repo_root))

prefer_ollama = not any(os.environ.get(key) for key in ("OPENAI_API_KEY", "GEMINI_API_KEY", "ANTHROPIC_API_KEY", "TOGETHER_API_KEY"))
if prefer_ollama:
    os.environ.setdefault("INDEX_LLM_PROVIDER", "ollama")
    os.environ.setdefault("INDEX_EMBEDDING_PROVIDER", "ollama")
os.environ.setdefault("OLLAMA_ENDPOINT", "http://127.0.0.1:11434")
os.environ.setdefault("MPLCONFIGDIR", str(repo_root / "output" / ".matplotlib"))

from eval.medqa_smoke import load_questions, format_options, extract_answer
from ingest.mimic import MimicDischargeSubsetConfig, extract_mimic_discharge_subset
from rag.index import ensure_index
from rag.retrieve import query_index_context
from llama_index.llms.ollama import Ollama
from rag.visualize import save_clinical_entity_graph, save_clinical_entity_graph_jpeg

# Edit these two lists to control the experiment grid. Each generation model is used
# both to build the property graph and to answer questions for that run.
generation_model_sweep = [
    # "gpt-oss:20b-cloud",
    # "gpt-oss:120b-cloud",
    # "gemma4",
    "qwen3.5:0.8b",
    # "qwen3:4b",
    # "qwen3:8b",
    "qwen3:1.7b",
    # "llama3:8b",
    "llama3.2:1b",
    "medgemma1.5",
    "gemma3:1b",
    "mistral",
]

embedding_model_sweep = [
    "qwen3-embedding",
    "embeddinggemma:300m",
]

def model_slug(value: object | None) -> str:
    text = "unknown" if value is None else str(value)
    return "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in text).strip("_") or "unknown"

def combo_slug(generation_model: str, embedding_model: str) -> str:
    return f"gen-{model_slug(generation_model)}__embed-{model_slug(embedding_model)}"

print("repo_root:", repo_root)
print("python:", sys.version.split()[0])
print("index_llm_provider:", os.environ.get("INDEX_LLM_PROVIDER") or os.environ.get("GENERATION_MODEL_PROVIDER") or "ollama")
print("index_embedding_provider:", os.environ.get("INDEX_EMBEDDING_PROVIDER") or os.environ.get("RAG_EMBEDDING_MODEL_PROVIDER") or "ollama")
print("generation_model_sweep:", generation_model_sweep)
print("embedding_model_sweep:", embedding_model_sweep)
print("total_combinations:", len(generation_model_sweep) * len(embedding_model_sweep))


repo_root: /Users/oluwatosinoso/Library/CloudStorage/OneDrive-hull.ac.uk/argumentation_schemes
python: 3.12.10
index_llm_provider: ollama
index_embedding_provider: ollama
generation_model_sweep: ['gpt-oss:20b-cloud', 'gpt-oss:120b-cloud', 'gemma4', 'qwen3.5:0.8b', 'qwen3:4b', 'qwen3:8b', 'qwen3:1.7b', 'llama3:8b', 'llama3.2:1b', 'medgemma1.5', 'gemma3:1b', 'mistral']
embedding_model_sweep: ['qwen3-embedding', 'embeddinggemma:300m']
total_combinations: 24


In [ ]:
# Source paths.
mimic_csv = repo_root / "data" / "mimic_iv_note" / "discharge.csv"
mimic_subset_dir = repo_root / "data" / "evidence" / "mimic_discharge_subset"
subset_limit = 20
subset_max_chars = 3000
test_jsonl_candidates = [
    repo_root / "test.jsonl",
    repo_root / "data" / "medqa" / "data_clean" / "questions" / "US" / "test.jsonl",
    repo_root / "data" / "eval" / "test.jsonl",
]
test_jsonl = next((path for path in test_jsonl_candidates if path.exists()), None)
output_dir = repo_root / "output"
output_dir.mkdir(parents=True, exist_ok=True)

use_umls = True
schema_guided = False

print("mimic_csv:", mimic_csv)
print("mimic_subset_dir:", mimic_subset_dir)
print("test_jsonl:", test_jsonl)
print("use_umls:", use_umls)
print("schema_guided:", schema_guided)


mimic_csv: /Users/oluwatosinoso/Library/CloudStorage/OneDrive-hull.ac.uk/argumentation_schemes/data/mimic_iv_note/discharge.csv
mimic_subset_dir: /Users/oluwatosinoso/Library/CloudStorage/OneDrive-hull.ac.uk/argumentation_schemes/data/evidence/mimic_discharge_subset
test_jsonl: /Users/oluwatosinoso/Library/CloudStorage/OneDrive-hull.ac.uk/argumentation_schemes/data/medqa/data_clean/questions/US/test.jsonl
use_umls: True
schema_guided: False


In [3]:
# Extract a small discharge-note subset used by every model combination.
if not mimic_csv.exists():
    raise FileNotFoundError(f"Missing MIMIC discharge CSV: {mimic_csv}")

extract_mimic_discharge_subset(
    MimicDischargeSubsetConfig(
        csv_path=mimic_csv,
        output_dir=mimic_subset_dir,
        limit=subset_limit,
        note_type="DS",
        max_chars=subset_max_chars,
        overwrite=True,
    )
)
print("Prepared MIMIC subset in:", mimic_subset_dir)


Prepared MIMIC subset in: /Users/oluwatosinoso/Library/CloudStorage/OneDrive-hull.ac.uk/argumentation_schemes/data/evidence/mimic_discharge_subset


In [ ]:
# Load a small test set.
if test_jsonl is not None:
    total_questions = sum(1 for line in test_jsonl.open("r", encoding="utf-8") if line.strip())
    sample_size = min(15, total_questions)
    questions = load_questions(test_jsonl, sample_size=sample_size)
else:
    questions = [
        {
            "id": f"demo-{i}",
            "question": "What is the preferred next step for a patient with CKD and uncontrolled hypertension?",
            "answer": "A",
            "options": {
                "A": "Optimize blood pressure control and renal protection",
                "B": "Stop all medication",
                "C": "Ignore the blood pressure",
            },
        }
        for i in range(1, 11)
    ]

pd.DataFrame([{"id": q.get("id"), "question": q.get("question"), "answer": q.get("answer")} for q in questions])


,id,question,answer
0,None,A 42-year-old woman with a history of depressi...,Sumatriptan
1,None,Eight weeks after starting a new weight-loss m...,Inhibition of lipase
2,None,A 27-year-old HIV positive female gave birth t...,Polymerase chain reaction
3,None,A 50-year-old man presents to his primary care...,Positive emission tomography (PET) of chest now
4,None,A 57-year-old homeless man is brought to the e...,Substance abuse
5,None,A 52-year-old man comes to the physician becau...,Fexofenadine
6,None,"A 28-year-old woman, gravida 2, para 1, at 40 ...",Treat and transfer the patient after she makes...
7,None,A 46-year-old man is brought to the emergency ...,Cardiac contusion
8,None,A 27-year-old man presents to the emergency ro...,Rotator cuff injury
9,None,A 68-year-old man presents with difficulty bre...,Papilledema


In [ ]:
# Build/query each generation-model x embedding-model combination and collect artifacts.
rows = []
artifact_rows = []
artifact_dir = output_dir / "hh_property_graph_artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)
ollama_endpoint = os.environ.get("OLLAMA_ENDPOINT", "http://127.0.0.1:11434")

for generation_model in generation_model_sweep:
    os.environ["INDEX_LLM_MODEL"] = generation_model
    for embedding_model in embedding_model_sweep:
        os.environ["INDEX_EMBEDDING_MODEL"] = embedding_model
        slug = combo_slug(generation_model, embedding_model)
        indexed_graph = await asyncio.to_thread(
            ensure_index,
            input_dir=mimic_subset_dir,
            output_dir=output_dir,
            use_umls=use_umls,
            schema_guided=schema_guided,
        )

        query_llm = Ollama(model=generation_model, base_url=ollama_endpoint)
        html_path = artifact_dir / f"{slug}.html"
        jpeg_path = artifact_dir / f"{slug}.jpg"
        results_path = artifact_dir / f"{slug}_results.csv"
        graph_html = save_clinical_entity_graph(output_dir, html_path, source="auto")
        graph_jpeg = save_clinical_entity_graph_jpeg(
            output_dir,
            jpeg_path,
            source="auto",
            title=f"Property graph: {slug}",
        )
        print("Saved graph artifacts:", graph_html, graph_jpeg)

        combo_rows = []
        for item in questions:
            prompt = item["question"]
            if item.get("options"):
                prompt = f"{prompt}\n\n" + format_options(item["options"])
            response, context = await query_index_context(index=indexed_graph, query=prompt, llm=query_llm)
            predicted = extract_answer(response, item["options"]) if item.get("options") else None
            combo_rows.append({
                "id": item.get("id"),
                "question": item["question"],
                "gold_answer": item.get("answer"),
                "predicted": predicted,
                "model": generation_model,
                "embedding_model": embedding_model,
                "combo_slug": slug,
                "response": response[:1000],
                "context_preview": context[:1000] if context else None,
                "html_plot": graph_html.as_posix(),
                "jpeg_plot": graph_jpeg.as_posix(),
            })

        combo_results = pd.DataFrame(combo_rows)
        combo_results.to_csv(results_path, index=False)
        rows.extend(combo_rows)
        artifact_rows.append({
            "combo_slug": slug,
            "model": generation_model,
            "embedding_model": embedding_model,
            "html_plot": graph_html.as_posix(),
            "jpeg_plot": graph_jpeg.as_posix(),
            "results_csv": results_path.as_posix(),
            "question_count": len(combo_results),
        })

results = pd.DataFrame(rows)
artifacts = pd.DataFrame(artifact_rows)
artifacts


LlamaIndex index is missing or incompatible:
  - source fingerprint changed


In [ ]:
# Persist combined notebook outputs.
graph_manifest = output_dir / "index_manifest.json"
results_path = output_dir / "hh_mimic_subset_property_graph_results.csv"
artifacts_path = output_dir / "hh_property_graph_artifacts_manifest.csv"
results.to_csv(results_path, index=False)
artifacts.to_csv(artifacts_path, index=False)
print(graph_manifest)
print(results_path)
print(artifacts_path)
results[["id", "combo_slug", "model", "embedding_model", "gold_answer", "predicted"]]
